In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("pipeline_1") \
    .getOrCreate()

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

In [0]:
# reading the data 
order_df=spark.read.csv('/Workspace/pipeline_first/notebook 1 for pipeline 1/pyspark_pipeline_orders.csv',inferSchema=True,header=True)
customer_df=spark.read.csv('/Workspace/pipeline_first/notebook 1 for pipeline 1/pyspark_pipeline_customers_scd.csv',inferSchema=True,header=True)

In [0]:
display(order_df.limit(5))
display(customer_df.limit(5))

order_id,customer_id,product_id,order_date,quantity,price,region,status
O100000,C00861,P001,2023-09-04,1,19.35,North,SHIPPED
O100001,C01295,P003,2023-03-09,5,9.59,South,PLACED
O100002,C01131,P002,2023-11-22,2,27.49,South,SHIPPED
O100003,C01096,P001,2023-11-30,1,21.44,Central,PLACED
O100004,C01639,P001,2023-07-15,1,20.44,Central,PLACED


customer_id,first_name,last_name,gender,email,address,region,effective_date,end_date,is_current,version
C02568,Jamie,Lopez,F,c02568@example.com,100 Main St,West,2023-01-01,null,true,1
C02561,Chris,Lee,F,c02561@example.com,101 Main St,North,2023-01-01,null,true,1
C01895,Alex,Lopez,F,c01895@example.com,102 Main St,West,2023-01-01,null,true,1
C00648,Morgan,Davis,F,c00648@example.com,103 Main St,West,2023-01-01,null,true,1
C00828,Chris,Davis,F,c00828@example.com,104 Main St,East,2023-01-01,2024-06-30,false,1


## bronze layer creation here 

In [0]:
bronze_orders='default.default.bronze_orders' # seting the location in the variable
bronze_customer='default.default.bronze_customer'
# mergeschema = True It automatically adds new columns from the new data into the existing table schema.
order_df.withColumn("ingest_time",current_timestamp())\
    .write.format("delta").mode('append').option("mergeSchema","True").saveAsTable(bronze_orders)

# data brick will check this table is already created or not in catalogs 
customer_df.withColumn("ingest_time",current_timestamp())\
    .write.format("delta").mode("overwrite").option("mergeschema","True").saveAsTable(bronze_customer)

print("bronze table are written ")

bronze table are written 


## bronze > silver layer

In [0]:
df=spark.read.table(bronze_orders)
w=Window.partitionBy("order_id").orderBy(desc("ingest_time"))
df=df.withColumn("rn",row_number().over(w))
deduped_df=df.filter(df.rn==1).drop("rn")
display(deduped_df.limit(5))
df=deduped_df # latest_data per day after overwrite

order_id,customer_id,product_id,order_date,quantity,price,region,status,ingest_time
O100000,C00861,P001,2023-09-04,1,19.35,North,SHIPPED,2026-03-07T14:10:58.863Z
O100001,C01295,P003,2023-03-09,5,9.59,South,PLACED,2026-03-07T14:10:58.863Z
O100002,C01131,P002,2023-11-22,2,27.49,South,SHIPPED,2026-03-07T14:10:58.863Z
O100003,C01096,P001,2023-11-30,1,21.44,Central,PLACED,2026-03-07T14:10:58.863Z
O100004,C01639,P001,2023-07-15,1,20.44,Central,PLACED,2026-03-07T14:10:58.863Z


bronze to silver

In [0]:
silver_order='default.default.silver_order' # location defining here
df_bronze_order=df # doing the tansformation here on the latest_Data with some transformation
df_orders_silver=(
  df_bronze_order.withColumn("order_date", to_date("order_date","yyyy-MM-dd"))\
    .withColumn('quantity',col('quantity').cast("int"))\
      .withColumn('price',col('price').cast("double"))\
        .filter(col('order_date').isNotNull())\
          .withColumn("sales_amount", col("quantity") * col("price"))\
            .filter(col("order_date").isNotNull())\
          .dropDuplicates(["order_id"])
)
df_orders_silver.write.format("delta").mode("overwrite").option("mergeSchema","True").saveAsTable(silver_order)
print("silver table are written ")

silver table are written 


silver to gold

## Bronze: raw ingested data
## 
## Silver: cleaned and validated data
## 
## Gold: aggregated business-ready data

In [0]:
gold_sales='default.default.gold_sales'
df_silver=spark.read.table(silver_order)
df_daily_sales=df_silver.groupBy('order_date','region','product_id').agg(sum('sales_amount').alias('sales_amount'),sum('quantity').alias('units'))

df_daily_sales.write.format("delta").mode("overwrite").option("mergeSchema","True").saveAsTable(gold_sales)
display(df_daily_sales.limit(5))
print("gold table is written ")

order_date,region,product_id,sales_amount,units
2023-09-04,North,P001,40.010000000000005,2
2023-03-09,South,P003,47.95,5
2023-11-22,South,P002,54.98,2
2023-11-30,Central,P001,63.0,3
2023-07-15,Central,P001,59.620000000000005,3


gold table is written 


## slowly changing dimensions

type 1

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# 1. Setup Table Paths
bronze_customer = 'default.default.bronze_customer'
dim_customer_table = 'default.default.scd1_customer'

# 2. Get the LATEST record for each customer from Bronze
# This fixes the "Drop Duplicates" mistake by ensuring we get the newest data
window_spec = Window.partitionBy("customer_id").orderBy(col("ingest_time").desc())

latest_updates = spark.read.table(bronze_customer) \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num") #droping this column after rank filter for the latest data 

# 3. Initialize the table ONLY if it doesn't exist
# This fixes the "Wipe-Out" mistake
if not spark.catalog.tableExists(dim_customer_table):
    print(f"Initializing {dim_customer_table}...")
    latest_updates.write.format("delta").mode("overwrite").saveAsTable(dim_customer_table)
else:
    # 4. Perform the surgical MERGE (SCD Type 1)
    print(f"Merging updates into {dim_customer_table}...")
    dim = DeltaTable.forName(spark, dim_customer_table)
    
    dim.alias("target").merge(
        latest_updates.alias("source"),
        "target.customer_id = source.customer_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

print("SCD1 Process Complete.")

Merging updates into default.default.scd1_customer...
SCD1 Process Complete.


## scd type 2

In [0]:
scd2_customer_table = 'default.default.scd2_customer'

# 1. Check if table exists
if not spark.catalog.tableExists(scd2_customer_table):
    # INITIALIZE
    initialize_dim_scd2 = latest_updates \
        .withColumn("effective_date", current_date()) \
        .withColumn("end_date", lit(None).cast("date")) \
        .withColumn("is_current", lit(True)) \
        .withColumn("version", lit(1))
    
    initialize_dim_scd2.write.format("delta").saveAsTable(scd2_customer_table)
    print("Table Initialized.")

else:
    # UPDATE/MERGE
    dim_table = DeltaTable.forName(spark, scd2_customer_table)
    
    # Step 1: Close the old records
    dim_table.alias("d").merge(
        updates_raw.alias("u"),
        "d.customer_id = u.customer_id AND d.is_current = true AND (d.address <> u.address OR d.region <> u.region)"
    ).whenMatchedUpdate(set = {
        "is_current" : "false",
        "end_date" : "u.effective_date"
    }).execute()

    # Step 2: Open the new records
    dim_table.alias("d").merge(
        updates_raw.alias("u"),
        "d.customer_id = u.customer_id AND d.is_current = true"
    ).whenNotMatchedInsertAll().execute()
    print("Table Merged.")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.sql(f"SELECT * FROM default.default.scd2_customer WHERE is_current = false ORDER BY customer_id ").head(10))

customer_id,first_name,last_name,gender,email,address,region,effective_date,end_date,is_current,version,ingest_time
C00067,Alex,Wilson,F,c00067@example.com,142 Main St,South,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00071,Jordan,Johnson,F,c00071@example.com,133 Main St,South,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00077,Taylor,Smith,F,c00077@example.com,112 Main St,South,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00204,Taylor,Johnson,M,c00204@example.com,125 Main St,North,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00397,Taylor,Lee,M,c00397@example.com,134 Main St,North,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00540,Casey,Lee,F,c00540@example.com,118 Main St,East,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00545,Jamie,Garcia,F,c00545@example.com,132 Main St,Central,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00560,Jordan,Johnson,F,c00560@example.com,108 Main St,South,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00648,Morgan,Davis,F,c00648@example.com,103 Main St,West,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z
C00671,Casey,Johnson,M,c00671@example.com,127 Main St,Central,2023-01-01,2024-07-01,false,1,2026-03-07T14:11:00.840Z


## scd type 3

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp

scd3_customer_table = 'default.default.scd3_customer'

# Initialize with current data and a Null previous column
initialize_dim_scd3 = latest_updates \
    .withColumn("current_address", col("address")) \
    .withColumn("previous_address", lit(None).cast("string")) \
    .withColumn("last_updated", current_timestamp())

initialize_dim_scd3.write.format("delta").mode("overwrite").saveAsTable(scd3_customer_table)

############################################################################
from delta.tables import DeltaTable

dim_type3 = DeltaTable.forName(spark, scd3_customer_table)

dim_type3.alias("target").merge(
    updates_raw.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(
    # Only update if the address is actually different
    condition = "target.current_address <> source.address",
    set = {
        "previous_address": "target.current_address",  # Move old to previous
        "current_address": "source.address",           # Set new to current
        "last_updated": "current_timestamp()"
    }
).whenNotMatchedInsert(values = {
    "customer_id": "source.customer_id",
    "current_address": "source.address",
    "previous_address": "CAST(NULL AS STRING)",  # FIXED: SQL string instead of lit()
    "last_updated": "current_timestamp()"         # FIXED: SQL string instead of function
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]